In [ ]:
!pip install -q "datasets<4" transformers accelerate jiwer librosa soundfile

In [ ]:
from datasets import load_dataset, Audio

N = 200
ds = load_dataset("google/fleurs", "ur_pk", split="test", trust_remote_code=True)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))
ds = ds.select(range(min(N, len(ds))))
print(ds)
print(ds[0]["raw_transcription"])

README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

parquet-data/ur_pk/train-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 1.60GB            

parquet-data/ur_pk/train-00000-of-00001.(…): downloading bytes:           |  0.00B            

parquet-data/ur_pk/validation-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  174MB            

parquet-data/ur_pk/validation-00000-of-0(…): downloading bytes:           |  0.00B            

...

parquet-data/ur_pk/test-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2109 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/267 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/299 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
    num_rows: 200
})
ان کے دور میں خفیف تلوُّث ایزا مسئلہ نہیں تھا جیسا کہ وہ آج ہے۔ ان کی سکونت ایسے شہروں یا حوزات میں ہے جہا ں تک بآسانی پہنچا جا سکتا ہے بہ نسبت ان مقامات کے جو ماڈرن زمانے میں تعمیر کئے گئے۔


In [ ]:
import torch
from transformers import pipeline

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=0 if torch.cuda.is_available() else -1,
    chunk_length_s=30,
    generate_kwargs={"language": "urdu", "task": "transcribe"},
)

audio_inputs = [{"raw": a["audio"]["array"], "sampling_rate": 16000} for a in ds]
outputs = asr(audio_inputs, batch_size=8)
hyps = [o["text"] for o in outputs]
refs = list(ds["raw_transcription"])

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (

In [ ]:
import re, jiwer

def norm(t):
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

refs_n = [norm(r) for r in refs]
hyps_n = [norm(h) for h in hyps]

print("WER:", round(jiwer.wer(refs_n, hyps_n) * 100, 1), "%")
print("CER:", round(jiwer.cer(refs_n, hyps_n) * 100, 1), "%")

WER: 36.6 %
CER: 13.8 %


In [ ]:
import pandas as pd

rows = []
for r, h in zip(refs_n, hyps_n):
    rows.append({"ref": r, "hyp": h,
                 "wer": jiwer.wer(r, h) if r else None,
                 "has_latin": bool(re.search(r"[a-zA-Z]", r)),
                 "has_digit": bool(re.search(r"\d", r))})
df = pd.DataFrame(rows)
df.to_csv("baseline_whisper_small_fleurs_ur.csv", index=False)

pd.set_option("display.max_colwidth", 120)
df.sort_values("wer", ascending=False).head(30)

,ref,hyp,wer,has_latin,has_digit
36,حکمران جماعت ساوتھ ویسٹ افریقہ پیپلز آرگنائزیشن ایس ڈبلیو اے پی او نے بھی پارلیمانی الیکشنز میں اپنی اکثرییت برقرار ...,حکمرانے جماعت ساوت ویسٹ اور فیقہ پیلز ارگانیزیشن ایر دویو ایپی اونے بھی پارلی مانی الیکشنز اپنی اکسریت برکرہ رکھی,0.761905,False,False
126,سکوٹرب بس 403 باقاعدگی سے کیبو ڈا روکا میں رکتے ہوئے سنٹارا جاتی ہے,اسکوٹر بس چاستیز بکائدی سے کیو بھی در روکہ میں رکتی ہوئے سنارٹہ کی جاتی ہے,0.714286,False,True
77,1537ء میں اسونسیون کی بنیاد کے بعد پیراگئے اپنا دیسی کردار اور شناخت برقرار رکھنے میں کامیاب رہا,پندرہ سو سیہتیس میں اونیس ویس کی بنیاد پیر آگئے اپنا دیسے کردار اور شناد کو برکرہ رکھنے میں کامیاب رہا,0.666667,False,True
149,انجام دیے گئے زیادہ تر کام نظریاتی تھے لیکن یہ پروگرام برج قوس کی کہکشاں سے بنے مشاہدات کے معیاری نمونہ کیلئے تحریر ...,انجام دیئے گئے زیادہ تر کام نظری آتی تھے لیکن وہ پر اکرام بڑھ جے کوس کی کہکشہ سے بنے میں شاہدات کے محال نمونے کے لئے...,0.653846,False,False
1,فلپائنی فوٹوگرافروں کے ساتھ چھےاغوا شدگان کو جلد رہا کر دیا گیا تھا ج ن میں بچ ے اور ب وڑھے شامل تھے,پیل پائنی فورٹو گرافروں کے ساتھ چھے اگوہ شدگان کو جل رہا کر دیا گیا تھا چین میں بچوں اور بڑے شاہوں ملتے,0.652174,False,False
128,سائنس کے تمام امور بشمول نفسیات پر ارسٹوٹل کے نظریے تسلیم کیے گئے تھے,سائنس کے تمام عمور وشمور نفسیات پر اس طوریٹ کے نظر یہ تصمیم کی گئے,0.642857,False,False
135,فوٹوگرفرس بعد میں اس بوڑھی عورت کی جگہ بیٹھ گئے جو رفع حاجت کے لئے گئی مینڈوزا کو گولی مار دی گئی,فورٹوگرافرز بعد میں اس پوڑی عورت کے جگہ بیٹکے جرفہ حجت کے لے گئی میں ڈوزوہ کھکولی ماردی گئی,0.636364,False,False
10,اٹلانٹک ہریکین سیزن کے دسواں نامی طوفان سب ٹراپیکل اسٹارم جیری آج بحراوقیانوس میں بن گیا,اٹلاڈنٹک ہرکن سیزن کے دستمہ نامی توفان سب ٹروپکال اسٹرون جوری آج بہرہ وکی آنوز میں بن گیا,0.625000,False,False
97,لوگ آج کل پیغامات کمپیوٹر اسکرین پر لکھتے ہیں انھیں کبھی شارپنر کے قریب بھی جانے کی ضرورت نہیں پڑتی,لوگ آجکل پیگامات کمپوٹر سکرن پر لیتے ہیں انہیں کبھی کبار شوپنر کے قریب بھی جانے کے طرح پٹ نہیں پڑتی,0.600000,False,False
138,پودے فوٹوسینتھیسزکے ذریعہ اپنا کھانا دھوپ میں بناتے ہیں وہ سایہ بھی مہیا کرتے ہیں,پوڈے فورٹو سن تھیسس کے ذریعے اپنا کھانا دھوپ میں بناتے ہیں وہ سایا بھی محاییہ گڑتے ہیں,0.600000,False,False


In [ ]:
print(df.groupby("has_latin")["wer"].agg(["count", "mean"]))
print(df.groupby("has_digit")["wer"].agg(["count", "mean"]))

           count      mean
has_latin                 
False        200  0.374625
           count      mean
has_digit                 
False        167  0.364219
True          33  0.427286


In [ ]:
import gc, time, torch, jiwer
from transformers import pipeline

try:
    del asr
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

def run_model(name, bs=2):
    asr = pipeline(
        "automatic-speech-recognition",
        model=name,
        device=0,
        torch_dtype=torch.float16,
        chunk_length_s=30,
        generate_kwargs={"language": "urdu", "task": "transcribe"},
    )
    # build a fresh list each time, because the pipeline consumes these dicts
    fresh_inputs = [{"raw": a["audio"]["array"], "sampling_rate": 16000} for a in ds]
    t = time.time()
    out = asr(fresh_inputs, batch_size=bs)
    secs = time.time() - t
    hyps_m = [norm(o["text"]) for o in out]
    del asr; gc.collect(); torch.cuda.empty_cache()
    return hyps_m, secs

results = {"openai/whisper-small": (jiwer.wer(refs_n, hyps_n), jiwer.cer(refs_n, hyps_n), None)}

for name in ["openai/whisper-medium", "openai/whisper-large-v3"]:
    h, secs = run_model(name)
    results[name] = (jiwer.wer(refs_n, h), jiwer.cer(refs_n, h), secs)
    print(name, "WER", round(results[name][0]*100, 1), "CER", round(results[name][1]*100, 1), "time", round(secs), "s")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


openai/whisper-medium WER 27.3 CER 9.6 time 528 s


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

...

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


openai/whisper-large-v3 WER 21.0 CER 7.5 time 790 s


In [ ]:
import pandas as pd
pd.DataFrame({"ref": refs_n, "hyp": h}).to_csv(
    "results_whisper_large_v3.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("results_whisper_large_v3.csv")

NameError: name 'refs_n' is not defined

In [1]:
!pip install -q "datasets<4" transformers accelerate jiwer librosa soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
pilot_refs = [
"دن میں تین بار پیراسیٹامول پانچ سو ملی گرام کی ایک گولی کھائیں",
"بروفین چار سو ملی گرام کی گولی کھانے کے بعد لیں",
"آگمنٹن چھ سو پچیس ملی گرام کی گولی صبح اور شام کھانے کے ساتھ لیں",
"ایموکسیلین دو سو پچاس ملی گرام کا شربت دن میں تین بار پانچ ملی لیٹر دیں",
"ازیتھرومائسن پانچ سو ملی گرام کی گولی روزانہ ایک بار تین دن تک لیں",
"میٹرونیڈازول چار سو ملی گرام کی گولی ہر آٹھ گھنٹے بعد لیں",
"میٹفارمن آٹھ سو پچاس ملی گرام کی گولی ناشتے اور رات کے کھانے کے ساتھ لیں",
"ایملوڈیپین پانچ ملی گرام کی گولی روزانہ صبح ایک بار لیں",
"ایٹورواسٹیٹن بیس ملی گرام کی گولی رات کو سونے سے پہلے لیں",
"پینٹوپرازول چالیس ملی گرام کی گولی ناشتے سے آدھا گھنٹہ پہلے لیں",
"اومیپرازول کا کیپسول خالی پیٹ صبح اٹھ کر لیں",
"سیپروفلوکساسن پانچ سو ملی گرام کی گولی دن میں دو بار سات دن تک کھائیں",
"ڈسپرین کی ایک گولی پانی میں گھول کر پئیں",
"کھانسی کا شربت رات کو سونے سے پہلے دو چمچ پئیں",
"اگر بخار ایک سو دو ڈگری سے زیادہ ہو تو فوراً ڈاکٹر کے پاس جائیں",
"بچے کو پیراسیٹامول کا شربت چھ گھنٹے بعد دس ملی لیٹر دیں",
"یہ دوا پندرہ دن تک مسلسل استعمال کریں اور بیچ میں نہ چھوڑیں",
"انسولین کی بیس یونٹ ناشتے سے پہلے لگائیں",
"سالبیوٹامول انہیلر کے دو پف ضرورت کے وقت لیں",
"وٹامن ڈی تھری کا ایک ساشے ہفتے میں ایک بار لیں",
"آنکھوں میں یہ قطرے دن میں چار بار ڈالیں",
"جلد پر یہ کریم رات کو سونے سے پہلے لگائیں",
"اگر الٹی یا چکر آئیں تو دوا بند کر کے ہمیں بتائیں",
"یہ گولی دودھ یا چائے کے ساتھ نہ لیں",
"لوسارٹن پچاس ملی گرام کی گولی روزانہ ایک بار لیں",
"وارفرین کی خوراک خون کے ٹیسٹ کی رپورٹ دیکھ کر بدلی جائے گی",
"ڈائیکلوفیناک پچاس ملی گرام کی گولی درد ہونے پر دن میں دو بار سے زیادہ نہ لیں",
"سیٹریزین دس ملی گرام کی گولی رات کو ایک بار لیں",
"پانچ دن تک روزانہ چالیس ملی گرام پریڈنیسولون لیں",
"فولک ایسڈ پانچ ملی گرام کی گولی روزانہ ایک بار لیں",
]
for i, s in enumerate(pilot_refs, 1):
    print(f"{i:02d}. {s}")

01. دن میں تین بار پیراسیٹامول پانچ سو ملی گرام کی ایک گولی کھائیں
02. بروفین چار سو ملی گرام کی گولی کھانے کے بعد لیں
03. آگمنٹن چھ سو پچیس ملی گرام کی گولی صبح اور شام کھانے کے ساتھ لیں
04. ایموکسیلین دو سو پچاس ملی گرام کا شربت دن میں تین بار پانچ ملی لیٹر دیں
05. ازیتھرومائسن پانچ سو ملی گرام کی گولی روزانہ ایک بار تین دن تک لیں
06. میٹرونیڈازول چار سو ملی گرام کی گولی ہر آٹھ گھنٹے بعد لیں
07. میٹفارمن آٹھ سو پچاس ملی گرام کی گولی ناشتے اور رات کے کھانے کے ساتھ لیں
08. ایملوڈیپین پانچ ملی گرام کی گولی روزانہ صبح ایک بار لیں
09. ایٹورواسٹیٹن بیس ملی گرام کی گولی رات کو سونے سے پہلے لیں
10. پینٹوپرازول چالیس ملی گرام کی گولی ناشتے سے آدھا گھنٹہ پہلے لیں
11. اومیپرازول کا کیپسول خالی پیٹ صبح اٹھ کر لیں
12. سیپروفلوکساسن پانچ سو ملی گرام کی گولی دن میں دو بار سات دن تک کھائیں
13. ڈسپرین کی ایک گولی پانی میں گھول کر پئیں
14. کھانسی کا شربت رات کو سونے سے پہلے دو چمچ پئیں
15. اگر بخار ایک سو دو ڈگری سے زیادہ ہو تو فوراً ڈاکٹر کے پاس جائیں
16. بچے کو پیراسیٹامول کا شربت چھ گھنٹے بعد دس مل

In [6]:
import re, glob, torch, jiwer, librosa
import pandas as pd
from google.colab import files
from transformers import pipeline

def norm(t):
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

paths = glob.glob("/content/*.mp4")
paths = sorted(paths, key=lambda p: int(re.match(r"\d+", p.split("/")[-1]).group()))
print(len(paths), "files found")
print([p.split("/")[-1] for p in paths])
assert len(paths) == len(pilot_refs), "File count doesn't match the 30 sentences"

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-large-v3",
    device=0,
    torch_dtype=torch.float16,
    chunk_length_s=30,
    generate_kwargs={"language": "urdu", "task": "transcribe"},
)

inputs = [{"raw": librosa.load(p, sr=16000)[0], "sampling_rate": 16000} for p in paths]
out = asr(inputs, batch_size=2)
hyps = [o["text"] for o in out]

refs_n = [norm(r) for r in pilot_refs]
hyps_n = [norm(h) for h in hyps]

print("Pilot WER:", round(jiwer.wer(refs_n, hyps_n) * 100, 1), "%")
print("Pilot CER:", round(jiwer.cer(refs_n, hyps_n) * 100, 1), "%")
print("Outputs containing Latin letters:", sum(bool(re.search(r"[a-zA-Z]", h)) for h in hyps_n), "/ 30")

df = pd.DataFrame({"file": [p.split("/")[-1] for p in paths], "ref": refs_n, "hyp": hyps_n})
df["wer"] = [jiwer.wer(r, h) for r, h in zip(refs_n, hyps_n)]
df.to_csv("pilot_results_large_v3.csv", index=False, encoding="utf-8-sig")
files.download("pilot_results_large_v3.csv")
df.sort_values("wer", ascending=False).head(10)

30 files found
['01.m4a.m4a.mp4', '02.m4a.mp4', '03.m4a.mp4', '4.m4a.mp4', '5.m4a.mp4', '6.m4a.mp4', '7.m4a.mp4', '8.m4a.mp4', '9.m4a.mp4', '10.m4a.mp4', '11.m4a.mp4', '12.m4a.mp4', '13.m4a.mp4', '14.m4a.mp4', '15.m4a.mp4', '16.m4a.mp4', '17.m4a.mp4', '18.m4a.mp4', '19.m4a.mp4', '20.m4a.mp4', '21.m4a.mp4', '22.m4a.mp4', '23.m4a.mp4', '24.m4a.mp4', '25.m4a.mp4', '26.m4a.mp4', '27.m4a.mp4', '28.m4a.mp4', '29.m4a.mp4', '30.m4a.mp4']


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

...

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
/tmp/ipykernel_525/4055936392.py:25: UserWarning: PySoundFile failed. Trying audioread instead.
  inputs = [{"raw": librosa.load(p, sr=16000)[0], "sampling_rate": 16000} for p in paths]
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_525/4055936392.py:25: User

Pilot WER: 25.1 %
Pilot CER: 9.6 %
Outputs containing Latin letters: 0 / 30


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,file,ref,hyp,wer
28,29.m4a.mp4,پانچ دن تک روزانہ چالیس ملی گرام پریڈنیسولون لیں,پانچ سن تک روزانہ چالیس میلی گرام پریٹنی سے لو...,0.555556
18,19.m4a.mp4,سالبیوٹامول انہیلر کے دو پف ضرورت کے وقت لیں,سال بیوٹی مال انیلر کے دو پاف ضرورت کے وقت لیں,0.555556
12,13.m4a.mp4,ڈسپرین کی ایک گولی پانی میں گھول کر پئیں,دسپرین کے گولی پانی میں گھول کر پیئیں,0.444444
9,10.m4a.mp4,پینٹوپرازول چالیس ملی گرام کی گولی ناشتے سے آد...,اپینٹو پارازول چالیس میل گرام کی گولی ناشتے سے...,0.416667
7,8.m4a.mp4,ایملوڈیپین پانچ ملی گرام کی گولی روزانہ صبح ای...,ایملو ٹپین پانچ میلی گرام کی گولی روزانہ صبح ا...,0.363636
8,9.m4a.mp4,ایٹورواسٹیٹن بیس ملی گرام کی گولی رات کو سونے ...,ایٹروس ٹیٹن 20 میلی گرام کی گولی رات کو سونے س...,0.333333
5,6.m4a.mp4,میٹرونیڈازول چار سو ملی گرام کی گولی ہر آٹھ گھ...,میٹران آئی ڈیزل چار سو میلی گرام کی گولی ہر آٹ...,0.333333
0,01.m4a.m4a.mp4,دن میں تین بار پیراسیٹامول پانچ سو ملی گرام کی...,دن میں تین بار پرسیٹومول پانچ سو میلی گرام کے ...,0.307692
13,14.m4a.mp4,کھانسی کا شربت رات کو سونے سے پہلے دو چمچ پئیں,کانسی کی شربت رات کو سونے سے پہلے دو چمچ پیئیں,0.272727
19,20.m4a.mp4,وٹامن ڈی تھری کا ایک ساشے ہفتے میں ایک بار لیں,وائٹمن دی تھی کا ایک ساشے ہفتے میں ایک بار لیں,0.272727


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
pilot_refs = [
"دن میں تین بار پیراسیٹامول پانچ سو ملی گرام کی ایک گولی کھائیں",
"بروفین چار سو ملی گرام کی گولی کھانے کے بعد لیں",
"آگمنٹن چھ سو پچیس ملی گرام کی گولی صبح اور شام کھانے کے ساتھ لیں",
"ایموکسیلین دو سو پچاس ملی گرام کا شربت دن میں تین بار پانچ ملی لیٹر دیں",
"ازیتھرومائسن پانچ سو ملی گرام کی گولی روزانہ ایک بار تین دن تک لیں",
"میٹرونیڈازول چار سو ملی گرام کی گولی ہر آٹھ گھنٹے بعد لیں",
"میٹفارمن آٹھ سو پچاس ملی گرام کی گولی ناشتے اور رات کے کھانے کے ساتھ لیں",
"ایملوڈیپین پانچ ملی گرام کی گولی روزانہ صبح ایک بار لیں",
"ایٹورواسٹیٹن بیس ملی گرام کی گولی رات کو سونے سے پہلے لیں",
"پینٹوپرازول چالیس ملی گرام کی گولی ناشتے سے آدھا گھنٹہ پہلے لیں",
"اومیپرازول کا کیپسول خالی پیٹ صبح اٹھ کر لیں",
"سیپروفلوکساسن پانچ سو ملی گرام کی گولی دن میں دو بار سات دن تک کھائیں",
"ڈسپرین کی ایک گولی پانی میں گھول کر پئیں",
"کھانسی کا شربت رات کو سونے سے پہلے دو چمچ پئیں",
"اگر بخار ایک سو دو ڈگری سے زیادہ ہو تو فوراً ڈاکٹر کے پاس جائیں",
"بچے کو پیراسیٹامول کا شربت چھ گھنٹے بعد دس ملی لیٹر دیں",
"یہ دوا پندرہ دن تک مسلسل استعمال کریں اور بیچ میں نہ چھوڑیں",
"انسولین کی بیس یونٹ ناشتے سے پہلے لگائیں",
"سالبیوٹامول انہیلر کے دو پف ضرورت کے وقت لیں",
"وٹامن ڈی تھری کا ایک ساشے ہفتے میں ایک بار لیں",
"آنکھوں میں یہ قطرے دن میں چار بار ڈالیں",
"جلد پر یہ کریم رات کو سونے سے پہلے لگائیں",
"اگر الٹی یا چکر آئیں تو دوا بند کر کے ہمیں بتائیں",
"یہ گولی دودھ یا چائے کے ساتھ نہ لیں",
"لوسارٹن پچاس ملی گرام کی گولی روزانہ ایک بار لیں",
"وارفرین کی خوراک خون کے ٹیسٹ کی رپورٹ دیکھ کر بدلی جائے گی",
"ڈائیکلوفیناک پچاس ملی گرام کی گولی درد ہونے پر دن میں دو بار سے زیادہ نہ لیں",
"سیٹریزین دس ملی گرام کی گولی رات کو ایک بار لیں",
"پانچ دن تک روزانہ چالیس ملی گرام پریڈنیسولون لیں",
"فولک ایسڈ پانچ ملی گرام کی گولی روزانہ ایک بار لیں",
]
for i, s in enumerate(pilot_refs, 1):
    print(f"{i:02d}. {s}")

01. دن میں تین بار پیراسیٹامول پانچ سو ملی گرام کی ایک گولی کھائیں
02. بروفین چار سو ملی گرام کی گولی کھانے کے بعد لیں
03. آگمنٹن چھ سو پچیس ملی گرام کی گولی صبح اور شام کھانے کے ساتھ لیں
04. ایموکسیلین دو سو پچاس ملی گرام کا شربت دن میں تین بار پانچ ملی لیٹر دیں
05. ازیتھرومائسن پانچ سو ملی گرام کی گولی روزانہ ایک بار تین دن تک لیں
06. میٹرونیڈازول چار سو ملی گرام کی گولی ہر آٹھ گھنٹے بعد لیں
07. میٹفارمن آٹھ سو پچاس ملی گرام کی گولی ناشتے اور رات کے کھانے کے ساتھ لیں
08. ایملوڈیپین پانچ ملی گرام کی گولی روزانہ صبح ایک بار لیں
09. ایٹورواسٹیٹن بیس ملی گرام کی گولی رات کو سونے سے پہلے لیں
10. پینٹوپرازول چالیس ملی گرام کی گولی ناشتے سے آدھا گھنٹہ پہلے لیں
11. اومیپرازول کا کیپسول خالی پیٹ صبح اٹھ کر لیں
12. سیپروفلوکساسن پانچ سو ملی گرام کی گولی دن میں دو بار سات دن تک کھائیں
13. ڈسپرین کی ایک گولی پانی میں گھول کر پئیں
14. کھانسی کا شربت رات کو سونے سے پہلے دو چمچ پئیں
15. اگر بخار ایک سو دو ڈگری سے زیادہ ہو تو فوراً ڈاکٹر کے پاس جائیں
16. بچے کو پیراسیٹامول کا شربت چھ گھنٹے بعد دس مل

In [7]:
df.to_csv("pilot_results_large_v3.csv", index=False, encoding="utf-8-sig")
from google.colab import files
files.download("pilot_results_large_v3.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>